In [1]:
#فرق ارور انالیسس این مسیله در مقایسه با پیش بینی قیمت مسکن 
#استفاده میکنیمConfusion Matrix از
#چون اینجا دو نوع اشتباه داریم:
#False Positive:
#مدل گفت زنده است، ولی مرده بود
#False Negative:
#مدل گفت مرده است، ولی زنده بود

In [2]:
import pandas as pd
df = pd.read_csv("train.csv")


# تغییر و یا حذف ستون ها /پرکردن دیتای خالی


df["Age"] = df["Age"].fillna(df["Age"].median())#برای پرکردن مقادیر خالی سن از median استفاده میکنیم 

df["Deck"] = df["Cabin"].str[0] #از ستون Cabin، اولین کاراکتر (حرف اول) را بردار و داخل یک ستون جدید به نام Deck ذخیره کن.
df["Deck"] = df["Deck"].fillna("U") #مقادیر خالی ستون جدید رو پر کنیم 
#هر مقدار خالی (NaN) در ستون Deck را با حرف "U" جایگزین کن.
df.drop("Cabin", axis=1, inplace=True)#حالا میتونیم ستون کابین رو کلا حذف کنیم


df["Embarked"] = df["Embarked"].fillna("S")#Embarked فقط دو مقدار خالی داشت.خیلی کم است، پس می‌توانیم با پرتکرارترین مقدار پر کنیم:


df["Title"] = df["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)
# Remove original Name column
df.drop("Name", axis=1, inplace=True)
title_counts = df["Title"].value_counts()
rare_titles = title_counts[title_counts < 10].index #هر عنوان کمتر از ۱۰ بار را پیدا می‌کند و تبدیل به Rare می‌کند.
df["Title"] = df["Title"].replace(rare_titles, "Rare")


df.drop("Ticket", axis=1, inplace=True)#فعلا ستون Ticket رو حذف میکنیم
# اطلاعات مستقیم قابل استفاده‌ای برای مدل ندارد.

#PassengerId معمولاً Feature خوبی نیست، چون فقط شماره ردیف مسافر است و اطلاعات واقعی درباره بقا نمی‌دهد. حذفش می‌کنیم:
df.drop("PassengerId", axis=1, inplace=True)

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1 #عدد یک واسه اینه که خود شخص رو هم بشماریم

#تبدیل ستون های متنی به عدد
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

df = pd.get_dummies(df, columns=["Embarked", "Title", "Deck"], drop_first=True)

In [3]:
X = df.drop("Survived", axis=1)
y = df["Survived"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2, #یعنی ۲۰٪ داده برای تست و ۸۰٪ برای آموزش.
    random_state=42
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_leaf": [1, 2, 4]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_
best_model.score(X_test, y_test)

0.8379888268156425

In [4]:
#حالا میریم سراغ ارور انالیسس میخوایم ببینیم مدل روی دیتای تست چقد خطا داره 

y_pred = best_model.predict(X_test)#اینجا مدل نهایی ما (best_model) 
#روی داده‌هایی که قبلاً ندیده (X_test) پیش‌بینی می‌کند

#errors = X_test[y_pred != y_test]#از جدول ویژگی‌های مسافرها (X_test) فقط ردیف‌هایی را بده که مدل اشتباه کرده.
#این کد رو کامنت کردم صرفا واسه این که بدونم میشه با این هم یه جدول نشون داده از داده هایی که مقادیر پیش بینی و واقعی دیتای تست برابر نیستن و مدل خطا داره

#بجای کد بالا یه کد میزنم واسه اینکه یه جدول بکشیم که سه تا ستون مقادیر پیش بینی و مقادیر واقعی و مقادیر خطا بهش اضافه شده تا بشه مقایسه کرد 

error_df = X_test.copy()#میخوایم یه دیتافریم داشته باشیم که یه تغییراتی بهش بدیم واسه همین کپی میگیریم از دیتای اول

error_df["Actual"] = y_test #سمت چپ مساوری داره میگه یه ستون به اسم اکتوال به دیتافریم ارور دی اف اضافه کن و مقادیر سمت راست مساوی رو بریز توش
error_df["Predicted"] = y_pred
error_df["Error"] = error_df["Actual"] != error_df["Predicted"]

error_df[error_df["Error"]]#ینی از دیتافریم ارور دی اف فقط ردیف هایی رو انتخاب کن که مقدار ستون ارور درانها ترو است 
#به این روش فیلترکردن در پانداز میگن بولین ایندکسینگ
#Indexing یعنی انتخاب یا دسترسی به بخشی از داده با استفاده از یک اندیس، نام یا شرط

#این کد فقط مسافرهایی را نشان می‌دهد که مدل اشتباه کرده.

,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,Embarked_Q,Embarked_S,Title_Miss,...,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_U,Actual,Predicted,Error
39,3,1,14.0,1,0,11.2417,2,False,False,True,...,False,False,False,False,False,False,True,1,0,True
621,1,0,42.0,1,0,52.5542,2,False,True,False,...,False,True,False,False,False,False,False,1,0,True
447,1,0,34.0,0,0,26.5500,1,False,True,False,...,False,False,False,False,False,False,True,1,0,True
192,3,1,19.0,1,0,7.8542,2,False,True,True,...,False,False,False,False,False,False,True,1,0,True
673,2,0,31.0,0,0,13.0000,1,False,True,False,...,False,False,False,False,False,False,True,1,0,True
235,3,1,28.0,0,0,7.5500,1,False,True,True,...,False,False,False,False,False,False,True,0,1,True
204,3,0,18.0,0,0,8.0500,1,False,True,False,...,False,False,False,False,False,False,True,1,0,True
23,1,0,28.0,0,0,35.5000,1,False,True,False,...,False,False,False,False,False,False,False,1,0,True
362,3,1,45.0,0,1,14.4542,2,False,False,False,...,False,False,False,False,False,False,True,0,1,True
772,2,1,57.0,0,0,10.5000,1,False,True,False,...,False,False,True,False,False,False,False,0,1,True


In [5]:
#در مسایل کلسیفیکیشن دو نوع خطا داریم: False Positive و False Negative

#1) False Positive مدل گفته زنده می‌ماند، ولی در واقع مرده:
#یعنی مدل زیادی خوش‌بین بوده.

false_positive = error_df[#تو دیفتا فریم ارور دی اف فقط اونایی رو نشون بده اکچوالشون صفره و پیش بینی شون 1 عه 
    (error_df["Actual"] == 0) &
    (error_df["Predicted"] == 1)
]

false_positive

,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,Embarked_Q,Embarked_S,Title_Miss,...,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_U,Actual,Predicted,Error
235,3,1,28.0,0,0,7.5500,1,False,True,True,...,False,False,False,False,False,False,True,0,1,True
362,3,1,45.0,0,1,14.4542,2,False,False,False,...,False,False,False,False,False,False,True,0,1,True
772,2,1,57.0,0,0,10.5000,1,False,True,False,...,False,False,True,False,False,False,False,0,1,True
312,2,1,26.0,1,1,26.0000,3,False,True,False,...,False,False,False,False,False,False,True,0,1,True
767,3,1,30.5,0,0,7.7500,1,True,False,True,...,False,False,False,False,False,False,True,0,1,True
357,2,1,38.0,0,0,13.0000,1,False,True,True,...,False,False,False,False,False,False,True,0,1,True
254,3,1,41.0,0,2,20.2125,3,False,True,False,...,False,False,False,False,False,False,True,0,1,True
292,2,0,36.0,0,0,12.8750,1,False,False,False,...,False,True,False,False,False,False,False,0,1,True
578,3,1,28.0,1,0,14.4583,2,False,False,False,...,False,False,False,False,False,False,True,0,1,True
49,3,1,18.0,1,0,17.8000,2,False,True,False,...,False,False,False,False,False,False,True,0,1,True


In [6]:
#2) False Negative مدل گفته می‌میرد، ولی در واقع زنده مانده:
#یعنی مدل نتوانسته بعضی نجات‌یافته‌ها را تشخیص دهد.
false_negative = error_df[
    (error_df["Actual"] == 1) &
    (error_df["Predicted"] == 0)
]

false_negative

,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,Embarked_Q,Embarked_S,Title_Miss,...,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_U,Actual,Predicted,Error
39,3,1,14.0,1,0,11.2417,2,False,False,True,...,False,False,False,False,False,False,True,1,0,True
621,1,0,42.0,1,0,52.5542,2,False,True,False,...,False,True,False,False,False,False,False,1,0,True
447,1,0,34.0,0,0,26.5500,1,False,True,False,...,False,False,False,False,False,False,True,1,0,True
192,3,1,19.0,1,0,7.8542,2,False,True,True,...,False,False,False,False,False,False,True,1,0,True
673,2,0,31.0,0,0,13.0000,1,False,True,False,...,False,False,False,False,False,False,True,1,0,True
204,3,0,18.0,0,0,8.0500,1,False,True,False,...,False,False,False,False,False,False,True,1,0,True
23,1,0,28.0,0,0,35.5000,1,False,True,False,...,False,False,False,False,False,False,False,1,0,True
712,1,0,48.0,1,0,52.0000,2,False,True,False,...,True,False,False,False,False,False,False,1,0,True
338,3,0,45.0,0,0,8.0500,1,False,True,False,...,False,False,False,False,False,False,True,1,0,True
286,3,0,30.0,0,0,9.5000,1,False,True,False,...,False,False,False,False,False,False,True,1,0,True


In [7]:
#حالا خطاها رو براساس فیچرهای اصلی مثلا روی sex فقط بررسی میکنیم
error_df[error_df["Error"]].groupby("Sex").size()# تو دیتافریم ارور دی اف اونایی که ستون ارورشون ترو هست فقط ستون فیچر سکس رو تعداد بده 
#می‌بینیم بیشتر اشتباه‌ها برای زن بوده یا مرد.
#زنا 1 و مردا 0 هستن

Sex
0    14
1    15
dtype: int64

In [8]:
#پس مدل تقریباً به یک اندازه برای زن و مرد خطا داشته
#منتهی بررسی نکرده چه تعداد کلا زن و چه تعداد کل مردها بوده 
#ینی ممکنه تعداد مثلا زنا بیشتر بوده ولی مدل تقرببا برابر با مردا خطا داشته درباره زنا
#پس برای اینکه درست بررسی کنیم که تو کدوم گروه مرد یا زن خطا بیشتر داشته نسبت 

error_df[error_df["Error"]].groupby("Sex").size() / X_test.groupby("Sex").size()#از هرگروه چند درصد اشتباه شده
#تعداد خطای هر گروه در ستون سکس رو تقسیم بر تعداد کل اون گروه میکنه
#groupby مقادیر یک ستون رو گروه بندی میکنه
#sizeتعداد هر گروه را می‌شمارد

Sex
0    0.127273
1    0.217391
dtype: float64

In [9]:
#نتیجه اینکه خطا روی زن ها بیشتر بوده

In [10]:
#خطای مدل روی فیچر Pclass رو بررسی میکنیم

error_df[error_df["Error"]].groupby("Pclass").size()
#می‌بینیم کدام کلاس بیشتر مشکل داشته.

Pclass
1     8
2     6
3    15
dtype: int64

In [11]:
#تعداد خطای کلاس 3 از همه بیشتر هست ولی فقط تعداد خام خطا کافی نیس چون تعداد افراد هر کلاس برابر نیس
#مثلاً اگر کلاس ۳ خیلی بیشتر مسافر داشته باشد، طبیعی است که تعداد خطای آن بیشتر باشد. باید نرخ خطا را ببینیم:
#پس به نسبت تعداد کل نفرات هر کلاس میایم خطای کلاس هارو بررسی میکنیم

error_rate_pclass = (
    error_df[error_df["Error"]].groupby("Pclass").size() /  X_test.groupby("Pclass").size()
)

error_rate_pclass

Pclass
1    0.150943
2    0.181818
3    0.161290
dtype: float64

In [12]:
#نتیجه اینکه تقریبا روی همه کلاس ها یه اندازه خطا داشته 

In [13]:
#نمونه‌های اشتباه را نگاه می‌کنیم.
error_df[error_df["Error"]][
    ["Sex", "Pclass", "Age", "Fare"]
]

,Sex,Pclass,Age,Fare
39,1,3,14.0,11.2417
621,0,1,42.0,52.5542
447,0,1,34.0,26.5500
192,1,3,19.0,7.8542
673,0,2,31.0,13.0000
235,1,3,28.0,7.5500
204,0,3,18.0,8.0500
23,0,1,28.0,35.5000
362,1,3,45.0,14.4542
772,1,2,57.0,10.5000


In [14]:
#بعد همه این کارها باید ببینیم
#1) آیا مشکل از داده (فیچر) است؟
#اول باید ببینی آیا اشتباه‌ها یک الگوی تکراری دارند یا تصادفی هستند.





In [15]:
#نتیجه فعلی Error Analysis:
#مشکل اصلی از Pclass نیست، چون نرخ خطاها نزدیک هم‌اند.
#احتمالاً باید بیشتر روی ترکیب‌ها نگاه کنیم، مثلاً: Sex + Pclass

(
    error_df[error_df["Error"]].groupby(["Sex", "Pclass"]).size()
    /
    X_test.groupby(["Sex", "Pclass"]).size()
)

Sex  Pclass
0    1         0.275862
     2         0.117647
     3         0.062500
1    1              NaN
     2         0.250000
     3         0.379310
dtype: float64

In [16]:
#بیشترین مشکل مدل: زنان کلاس ۳
#مدل برای این گروه گیج می‌شود، چون زنان کلاس ۳ رفتار یکسانی ندارند:
#بعضی زنده ماندند. بعضی مردند تفاوت‌های دیگری مثل خانواده، سن، کرایه، همراهان و... مهم می‌شوند
# مردان کلاس ۱ هم نرخ خطای بالایی دارند احتمالاً چون تعداد این گروه کم است و نمونه‌های خاص زیاد هستند 
#NaN برای زنان کلاس ۱ یعنی در X_test یا خطایی از این گروه نداشتی یا تقسیم بر صفر اتفاق افتاده (یعنی گروهی برای محاسبه وجود نداشته)

#آیا این یعنی باید فوراً فیچر جدید بسازیم؟
#نه. اول باید ببینیم این گروه‌های اشتباه چه ویژگی مشترکی دارند.
#مثلاً زنان کلاس ۳ که مدل اشتباه کرده‌اند رو بررسی میکنم ببینم دیگه چه ویژگی ای دارند

error_df[
    (error_df["Sex"] == 1) &
    (error_df["Pclass"] == 3) &
    (error_df["Error"])
]

,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,Embarked_Q,Embarked_S,Title_Miss,...,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_U,Actual,Predicted,Error
39,3,1,14.0,1,0,11.2417,2,False,False,True,...,False,False,False,False,False,False,True,1,0,True
192,3,1,19.0,1,0,7.8542,2,False,True,True,...,False,False,False,False,False,False,True,1,0,True
235,3,1,28.0,0,0,7.5500,1,False,True,True,...,False,False,False,False,False,False,True,0,1,True
362,3,1,45.0,0,1,14.4542,2,False,False,False,...,False,False,False,False,False,False,True,0,1,True
767,3,1,30.5,0,0,7.7500,1,True,False,True,...,False,False,False,False,False,False,True,0,1,True
254,3,1,41.0,0,2,20.2125,3,False,True,False,...,False,False,False,False,False,False,True,0,1,True
578,3,1,28.0,1,0,14.4583,2,False,False,False,...,False,False,False,False,False,False,True,0,1,True
49,3,1,18.0,1,0,17.8000,2,False,True,False,...,False,False,False,False,False,False,True,0,1,True
657,3,1,32.0,1,1,15.5000,3,True,False,False,...,False,False,False,False,False,False,True,0,1,True
852,3,1,9.0,1,1,15.2458,3,False,False,True,...,False,False,False,False,False,False,True,0,1,True


In [17]:
#چیزی که من دیدم از دیتایی که مدل اشتباه پیش بینی کرده با فیلتر زن هایی که از کلاس 3 هستن همشون بدون کابین بودن
#اینکه مدل اشتباه پیش بینی کرده واسه اینه که واقعا هم طبیعیه زن های طبقه فقیر کابین نداشته باشن و آمار مرگشون هم بالا باشه 
#ولی مدل به اشتباه گفته نجات پیدا میکنن و در واقع مرده بودن

In [18]:
#خب حالا ما میخوایم بازم یه فیلتر روی دیتافریم ارور اف دی بزنیم اونم فیلتر کردن زنای کلاس 3 هست که مدل اشتباه پیش بینی کرده نجاتشون رو

female_class3_errors = error_df[
    (error_df["Sex"] == 1) &
    (error_df["Pclass"] == 3) &
    (error_df["Error"])
]
female_class3_errors
#اگه میخواستیم فقط یه سری ستون (فیچر) خاص رو نشونمون بده این کدش بود

#error_df[
  #  (error_df["Sex"] == 1) &
  #  (error_df["Pclass"] == 3) &
   # (error_df["Error"])
#][
#    ["Age","Fare","SibSp","Parch","Title_Miss","Actual","Predicted"]
#]

,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,Embarked_Q,Embarked_S,Title_Miss,...,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_U,Actual,Predicted,Error
39,3,1,14.0,1,0,11.2417,2,False,False,True,...,False,False,False,False,False,False,True,1,0,True
192,3,1,19.0,1,0,7.8542,2,False,True,True,...,False,False,False,False,False,False,True,1,0,True
235,3,1,28.0,0,0,7.5500,1,False,True,True,...,False,False,False,False,False,False,True,0,1,True
362,3,1,45.0,0,1,14.4542,2,False,False,False,...,False,False,False,False,False,False,True,0,1,True
767,3,1,30.5,0,0,7.7500,1,True,False,True,...,False,False,False,False,False,False,True,0,1,True
254,3,1,41.0,0,2,20.2125,3,False,True,False,...,False,False,False,False,False,False,True,0,1,True
578,3,1,28.0,1,0,14.4583,2,False,False,False,...,False,False,False,False,False,False,True,0,1,True
49,3,1,18.0,1,0,17.8000,2,False,True,False,...,False,False,False,False,False,False,True,0,1,True
657,3,1,32.0,1,1,15.5000,3,True,False,False,...,False,False,False,False,False,False,True,0,1,True
852,3,1,9.0,1,1,15.2458,3,False,False,True,...,False,False,False,False,False,False,True,0,1,True


In [19]:
# بررسی ویژگی‌ها:
# - Age: بیشتر خطاها در بازه 18 تا 40 سال بودند (73%) اما الگوی قطعی دیده نشد.
# - SibSp: حدود 55% دارای یک همراه و 45% بدون همراه بودند، پس الگوی مشخصی ندارد.
# - Parch: حدود 64% بدون والد/فرزند همراه بودند، اما کافی نیست.
# - Fare: بیشتر مقادیر بین 10 تا 20 یا کمتر از 10 بودند، ولی الگوی قوی مشخص نشد.
# نتیجه:
# از سایر Featureها الگوی واضحی برای ساخت Feature جدید پیدا نشد.

In [20]:
X.columns

Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize',
       'Embarked_Q', 'Embarked_S', 'Title_Miss', 'Title_Mr', 'Title_Mrs',
       'Title_Rare', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F',
       'Deck_G', 'Deck_T', 'Deck_U'],
      dtype='str')

In [21]:
female_class3_errors[
    female_class3_errors["Actual"] == 1
]

,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,Embarked_Q,Embarked_S,Title_Miss,...,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_U,Actual,Predicted,Error
39,3,1,14.0,1,0,11.2417,2,False,False,True,...,False,False,False,False,False,False,True,1,0,True
192,3,1,19.0,1,0,7.8542,2,False,True,True,...,False,False,False,False,False,False,True,1,0,True
25,3,1,38.0,1,5,31.3875,7,False,True,False,...,False,False,False,False,False,False,True,1,0,True


In [22]:
female_class3_errors[
    female_class3_errors["Actual"] == 0
]

,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,Embarked_Q,Embarked_S,Title_Miss,...,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_U,Actual,Predicted,Error
235,3,1,28.0,0,0,7.5500,1,False,True,True,...,False,False,False,False,False,False,True,0,1,True
362,3,1,45.0,0,1,14.4542,2,False,False,False,...,False,False,False,False,False,False,True,0,1,True
767,3,1,30.5,0,0,7.7500,1,True,False,True,...,False,False,False,False,False,False,True,0,1,True
254,3,1,41.0,0,2,20.2125,3,False,True,False,...,False,False,False,False,False,False,True,0,1,True
578,3,1,28.0,1,0,14.4583,2,False,False,False,...,False,False,False,False,False,False,True,0,1,True
49,3,1,18.0,1,0,17.8000,2,False,True,False,...,False,False,False,False,False,False,True,0,1,True
657,3,1,32.0,1,1,15.5000,3,True,False,False,...,False,False,False,False,False,False,True,0,1,True
852,3,1,9.0,1,1,15.2458,3,False,False,True,...,False,False,False,False,False,False,True,0,1,True


In [23]:
female_class3_errors.groupby("FamilySize")["Actual"].value_counts()

FamilySize  Actual
1           0         2
2           0         3
            1         2
3           0         3
7           1         1
Name: count, dtype: int64